# Day 049 — Exercise 4: classification_metrics

**What you'll build:** `classification_metrics(model, X_test, y_test) -> dict` — compute accuracy, precision, recall, F1, confusion matrix, and the full classification report for a fitted classifier.

**Why it matters:** Accuracy alone is misleading when classes are imbalanced. A model that always predicts 'pass' is 65% accurate on our dataset — but it never identifies a failing student. Precision tells you how many predicted positives are correct; recall tells you how many actual positives were found; F1 is the harmonic mean of both.

## Provided: Setup + cross_validate_model + overfitting_report + train_classifier

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Numeric-only housing dataset (area, bedrooms, age → price)."""
    rng = np.random.default_rng(seed)
    area     = rng.uniform(500, 3000, n).round(0)
    bedrooms = rng.integers(1, 6, n)
    age      = rng.uniform(0, 50, n).round(1)
    price    = (area * 150 + bedrooms * 10_000 - age * 1_000
                + rng.standard_normal(n) * 10_000).round(-2)
    return pd.DataFrame({'area': area.astype(int), 'bedrooms': bedrooms,
                         'age': age, 'price': price.astype(int)})


def make_classification_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Student exam dataset: hours_studied + hours_sleep → passed (0/1)."""
    rng          = np.random.default_rng(seed)
    hours_studied = rng.uniform(0, 10, n).round(1)
    hours_sleep   = rng.uniform(4, 10, n).round(1)
    noise         = rng.standard_normal(n)
    score         = 1.5 * hours_studied + 0.5 * hours_sleep + noise
    passed        = (score > 9.0).astype(int)
    return pd.DataFrame({'hours_studied': hours_studied,
                         'hours_sleep':   hours_sleep,
                         'passed':        passed})


def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """K-fold cross-validation returning per-fold scores and summary stats."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    return {
        'scores':   scores,
        'mean':     round(float(scores.mean()), 4),
        'std':      round(float(scores.std()),  4),
        'min':      round(float(scores.min()),  4),
        'max':      round(float(scores.max()),  4),
        'cv_folds': cv,
        'scoring':  scoring,
    }


def overfitting_report(X: pd.DataFrame, y: pd.Series,
                        max_depths=range(1, 11),
                        test_size: float = 0.2,
                        random_state: int = 42) -> pd.DataFrame:
    """Train DecisionTreeRegressors at each depth; return train vs test R² table."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    records = []
    for depth in max_depths:
        m = DecisionTreeRegressor(max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        tr_r2 = float(r2_score(y_train, m.predict(X_train)))
        te_r2 = float(r2_score(y_test,  m.predict(X_test)))
        gap   = round(tr_r2 - te_r2, 4)
        records.append({
            'max_depth': depth,
            'train_r2':  round(tr_r2, 4),
            'test_r2':   round(te_r2, 4),
            'gap':       gap,
            'overfit':   bool(gap > 0.1),
        })
    return pd.DataFrame(records)


def train_classifier(X_train: pd.DataFrame,
                     y_train: pd.Series,
                     max_iter: int = 1000) -> LogisticRegression:
    """Fit LogisticRegression and return the fitted model."""
    model = LogisticRegression(random_state=42, max_iter=max_iter)
    model.fit(X_train, y_train)
    return model

## Your Implementation

In [ ]:
def classification_metrics(model, X_test: pd.DataFrame,
                            y_test: pd.Series) -> dict:
    """
    Full classification evaluation for a fitted model.

    Returns dict with keys:
        accuracy         — fraction of correct predictions
        precision        — TP / (TP + FP), zero_division=0
        recall           — TP / (TP + FN), zero_division=0
        f1               — harmonic mean of precision and recall
        confusion_matrix — 2D numpy array [[TN, FP], [FN, TP]]
        report           — sklearn classification_report string
    """
    y_pred = model.predict(X_test)
    # TODO: return {
    #     'accuracy':         round(float(accuracy_score(y_test, y_pred)), 4),
    #     'precision':        round(float(precision_score(y_test, y_pred,
    #                                                      zero_division=0)), 4),
    #     'recall':           round(float(recall_score(y_test, y_pred,
    #                                                   zero_division=0)), 4),
    #     'f1':               round(float(f1_score(y_test, y_pred,
    #                                               zero_division=0)), 4),
    #     'confusion_matrix': confusion_matrix(y_test, y_pred),
    #     'report':           classification_report(y_test, y_pred),
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df_c = make_classification_data(200)
    X_c  = df_c.drop(columns=['passed'])
    y_c  = df_c['passed']
    X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
        X_c, y_c, test_size=0.2, random_state=42
    )
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr_c), columns=X_c.columns)
    X_te_s = pd.DataFrame(scaler.transform(X_te_c),     columns=X_c.columns)
    clf    = train_classifier(X_tr_s, y_tr_c)

    # Check 1: defined, returns dict
    try:
        assert 'classification_metrics' in globals()
        result = classification_metrics(clf, X_te_s, y_te_c)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: classification_metrics returns dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all 6 keys present
    try:
        for k in ('accuracy', 'precision', 'recall', 'f1',
                  'confusion_matrix', 'report'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: all 6 keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: numeric metrics in [0, 1]
    try:
        for k in ('accuracy', 'precision', 'recall', 'f1'):
            v = result[k]
            assert 0.0 <= v <= 1.0, f'{k}={v} is not in [0, 1]'
        passed += 1; print(f'\u2705 Check 3: all numeric metrics in [0, 1]')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: confusion_matrix is 2D array with shape (2, 2)
    try:
        cm = result['confusion_matrix']
        assert hasattr(cm, 'shape'), 'confusion_matrix must be numpy array'
        assert cm.shape == (2, 2), \
            f'confusion_matrix shape {cm.shape} should be (2, 2)'
        assert cm.sum() == len(y_te_c), \
            f'confusion_matrix sum={cm.sum()} should equal n_test={len(y_te_c)}'
        passed += 1; print(f'\u2705 Check 4: confusion_matrix shape {cm.shape}, sum={cm.sum()}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: report is a non-empty string
    try:
        rep = result['report']
        assert isinstance(rep, str) and len(rep) > 20, \
            f'report must be a non-empty string, got {type(rep).__name__}'
        passed += 1; print(f'\u2705 Check 5: report is a {len(rep)}-char string')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def classification_metrics(model, X_test: pd.DataFrame,
                            y_test: pd.Series) -> dict:
    """Full classification evaluation: accuracy, precision, recall, F1, matrix, report."""
    y_pred = model.predict(X_test)
    return {
        'accuracy':         round(float(accuracy_score(y_test, y_pred)), 4),
        'precision':        round(float(precision_score(y_test, y_pred,
                                                         zero_division=0)), 4),
        'recall':           round(float(recall_score(y_test, y_pred,
                                                      zero_division=0)), 4),
        'f1':               round(float(f1_score(y_test, y_pred,
                                                  zero_division=0)), 4),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'report':           classification_report(y_test, y_pred),
    }
```

</details>